It groups individual players into their national teams, calculates their Live Attack Form (Shots on Target) and Live Defense Form (Goalkeeper Save %), and uses it to mathematically boost or penalize our original Expected Goals (xG) model!

In [13]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from numpy.matrixlib import defmatrix

In [14]:
df_live = pd.read_csv("../data/raw/2026_live_players_stats.csv")
df_live.head()


,player,team,team_country,position,age,birth_year,club,games,games_starts,minutes,...,gk_wins,gk_ties,gk_losses,gk_clean_sheets,gk_clean_sheets_pct,gk_pens_att,gk_pens_allowed,gk_pens_saved,gk_pens_missed,gk_pens_save_pct
0,Achref Abada,Algeria,Algeria,DF,27-010,1999,NaN,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Adil Boulbina,Algeria,Algeria,FW,23-054,2003,NaN,1,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Amine Gouiri,Algeria,Algeria,FW,26-129,2000,Marseille,2,2,148.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Anis Hadj Moussa,Algeria,Algeria,MF,24-134,2002,Feyenoord,2,1,78.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aïssa Mandi,Algeria,Algeria,DF,34-246,1991,Lille,2,2,180.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# FIX: Rename columns if they are cut off in your specific CSV
df_live.rename(columns={
    'shots_on_': 'shots_on', 
    'gk_save_p': 'gk_save_pct'
}, inplace=True, errors='ignore') 
# (errors='ignore' means it will only rename them if it needs to!)

In [16]:
# Find all columns that have 'shot' or 'save' in their name
shot_cols = [col for col in df_live.columns if 'shot' in col.lower()]
save_cols = [col for col in df_live.columns if 'save' in col.lower()]

print("Exact 'Shots' columns in your data:", shot_cols)
print("Exact 'Saves' columns in your data:", save_cols)

Exact 'Shots' columns in your data: ['shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target', 'gk_shots_on_target_against']
Exact 'Saves' columns in your data: ['gk_saves', 'gk_save_pct', 'gk_pens_saved', 'gk_pens_save_pct']


In [17]:
# ---------------------------------------------------------
# 1. LOAD OUR ORIGINAL POISSON MODEL
# ---------------------------------------------------------
df_master = pd.read_csv('../data/final/master_training_data.csv')

# Handle missing data for the model training
df_master['home_ea_score'] = df_master['home_ea_score'].fillna(60)
df_master['away_ea_score'] = df_master['away_ea_score'].fillna(60)
df_master['home_tm_value'] = df_master['home_tm_value'].fillna(1000000)
df_master['away_tm_value'] = df_master['away_tm_value'].fillna(1000000)

home_data = pd.DataFrame({'team': df_master['home_team'], 'opponent': df_master['away_team'], 'goals': df_master['home_score'], 'is_home_advantage': np.where(df_master['neutral'] == True, 0, 1), 'rank_diff': df_master['away_fifa_rank'] - df_master['home_fifa_rank'], 'ea_diff': df_master['home_ea_score'] - df_master['away_ea_score'], 'tm_log_diff': np.log1p(df_master['home_tm_value']) - np.log1p(df_master['away_tm_value'])})
away_data = pd.DataFrame({'team': df_master['away_team'], 'opponent': df_master['home_team'], 'goals': df_master['away_score'], 'is_home_advantage': 0, 'rank_diff': df_master['home_fifa_rank'] - df_master['away_fifa_rank'], 'ea_diff': df_master['away_ea_score'] - df_master['home_ea_score'], 'tm_log_diff': np.log1p(df_master['away_tm_value']) - np.log1p(df_master['home_tm_value'])})
goal_data = pd.concat([home_data, away_data], ignore_index=True).dropna(subset=['goals'])

# Fit the Model
poisson_model = smf.glm(formula="goals ~ is_home_advantage + rank_diff + ea_diff + tm_log_diff", data=goal_data, family=sm.families.Poisson()).fit()

# Load latest baseline stats
latest_ea = pd.read_csv('../data/processed/ea_squad_strength.csv').groupby('country').tail(1).set_index('country')['ea_squad_score'].to_dict()
latest_tm = pd.read_csv('../data/processed/tm_squad_value.csv').groupby('country').tail(1).set_index('country')['tm_squad_value_eur'].to_dict()
latest_fifa = pd.read_csv('../data/processed/fifa_rankings.csv').groupby('country').tail(1).set_index('country')['fifa_rank'].to_dict()

# ---------------------------------------------------------
# 2. LOAD & AGGREGATE LIVE 2026 TOURNAMENT DATA
# ---------------------------------------------------------
# # UPDATE THIS PATH TO YOUR EXACT CSV NAME!
# live_data_path = '../data/raw/live_2026_players.csv' 
# df_live = pd.read_csv(live_data_path)

# Aggregate Attack: Total Goals, Assists, and Shots on Target (USING CORRECT COLUMN NAME)
team_attack = df_live.groupby('team')[['goals', 'assists', 'shots_on_target']].sum().reset_index()

# Aggregate Defense: Goalkeeper Save Percentage
df_live['gk_save_pct'] = pd.to_numeric(df_live['gk_save_pct'], errors='coerce')
team_defense = df_live.dropna(subset=['gk_save_pct']).groupby('team')['gk_save_pct'].max().reset_index()

# Merge Live Stats
live_form = pd.merge(team_attack, team_defense, on='team', how='left')

# Calculate Tournament Averages to compare teams against
avg_shots_on = live_form['shots_on_target'].mean()
avg_save_pct = live_form['gk_save_pct'].mean()
if pd.isna(avg_save_pct): avg_save_pct = 70.0 # Default if missing

# Create dictionaries for fast lookup
live_shots = live_form.set_index('team')['shots_on_target'].to_dict()
live_saves = live_form.set_index('team')['gk_save_pct'].fillna(avg_save_pct).to_dict()

# ---------------------------------------------------------
# 3. LIVE KNOCKOUT PREDICTOR FUNCTION
# ---------------------------------------------------------
def predict_knockout_match(team1, team2, team1_home_adv=False, team2_home_adv=False):
    print(f"=====================================================")
    print(f"🏆 2026 WORLD CUP KNOCKOUT PREDICTION: {team1} vs {team2}")
    print(f"=====================================================")
    
    # 1. Get Baseline Stats
    t1_rank, t2_rank = latest_fifa.get(team1, 50), latest_fifa.get(team2, 50)
    t1_ea, t2_ea = latest_ea.get(team1, 70), latest_ea.get(team2, 70)
    t1_tm, t2_tm = latest_tm.get(team1, 50000000), latest_tm.get(team2, 50000000)
    
    # 2. Get Live Tournament Form
    t1_shots, t2_shots = live_shots.get(team1, avg_shots_on), live_shots.get(team2, avg_shots_on)
    t1_save_pct, t2_save_pct = live_saves.get(team1, avg_save_pct), live_saves.get(team2, avg_save_pct)
    
    print(f"📊 LIVE TOURNAMENT FORM:")
    print(f" - {team1}: {t1_shots} Shots on Target | GK Save Rate: {t1_save_pct:.1f}%")
    print(f" - {team2}: {t2_shots} Shots on Target | GK Save Rate: {t2_save_pct:.1f}% \n")
    
    # 3. Calculate Base Expected Goals (using our trained AI)
    t1_input = pd.DataFrame({'is_home_advantage': [1 if team1_home_adv else 0], 'rank_diff': [t2_rank - t1_rank], 'ea_diff': [t1_ea - t2_ea], 'tm_log_diff': [np.log1p(t1_tm) - np.log1p(t2_tm)]})
    t2_input = pd.DataFrame({'is_home_advantage': [1 if team2_home_adv else 0], 'rank_diff': [t1_rank - t2_rank], 'ea_diff': [t2_ea - t1_ea], 'tm_log_diff': [np.log1p(t2_tm) - np.log1p(t1_tm)]})
    
    base_xG_t1 = poisson_model.predict(t1_input).values[0]
    base_xG_t2 = poisson_model.predict(t2_input).values[0]
    
    # 4. Apply Live Form Multipliers!
    # If a team shoots better than average, they get a boost.
    t1_attack_mod = max(0.5, min(1.5, t1_shots / avg_shots_on))
    t1_defense_mod = max(0.5, min(1.5, (100 - t2_save_pct) / (100 - avg_save_pct)))
    
    t2_attack_mod = max(0.5, min(1.5, t2_shots / avg_shots_on))
    t2_defense_mod = max(0.5, min(1.5, (100 - t1_save_pct) / (100 - avg_save_pct)))
    
    final_xG_t1 = base_xG_t1 * t1_attack_mod * t1_defense_mod
    final_xG_t2 = base_xG_t2 * t2_attack_mod * t2_defense_mod
    
    print(f"⚽ BASELINE xG (Pre-Tournament): {team1} {base_xG_t1:.2f} | {team2} {base_xG_t2:.2f}")
    print(f"🔥 FINAL xG (Adjusted for Live Form): {team1} {final_xG_t1:.2f} | {team2} {final_xG_t2:.2f}\n")
    
    # 5. Determine the Winner
    if final_xG_t1 > final_xG_t2 + 0.15:
        print(f"🔮 PREDICTION: {team1} advances to the next round! 🏆")
    elif final_xG_t2 > final_xG_t1 + 0.15:
        print(f"🔮 PREDICTION: {team2} advances to the next round! 🏆")
    else:
        print(f"🔮 PREDICTION: Extremely tight match. Likely going to Extra Time / Penalties! 🤝")
        print(f"   (Slight mathematical edge to {'Team 1: ' + team1 if final_xG_t1 > final_xG_t2 else 'Team 2: ' + team2})")

# TEST IT WITH REAL TEAMS!
predict_knockout_match("France", "Netherlands")
print("\n")
predict_knockout_match("United States", "Colombia", team1_home_adv=True)

🏆 2026 WORLD CUP KNOCKOUT PREDICTION: France vs Netherlands
📊 LIVE TOURNAMENT FORM:
 - France: 13.0 Shots on Target | GK Save Rate: 66.7%
 - Netherlands: 13.0 Shots on Target | GK Save Rate: 72.7% 

⚽ BASELINE xG (Pre-Tournament): France 1.01 | Netherlands 1.14
🔥 FINAL xG (Adjusted for Live Form): France 1.12 | Netherlands 1.55

🔮 PREDICTION: Netherlands advances to the next round! 🏆


🏆 2026 WORLD CUP KNOCKOUT PREDICTION: United States vs Colombia
📊 LIVE TOURNAMENT FORM:
 - United States: 8.0 Shots on Target | GK Save Rate: 66.7%
 - Colombia: 13.0 Shots on Target | GK Save Rate: 66.7% 

⚽ BASELINE xG (Pre-Tournament): United States 1.58 | Colombia 0.99
🔥 FINAL xG (Adjusted for Live Form): United States 1.32 | Colombia 1.35

🔮 PREDICTION: Extremely tight match. Likely going to Extra Time / Penalties! 🤝
   (Slight mathematical edge to Team 2: Colombia)
